In [25]:
!pip install sentence-transformers rank-bm25 google-generativeai groq python-dotenv --quiet

In [26]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai
from groq import Groq
import os
import getpass
import warnings

warnings.filterwarnings("ignore")

In [27]:
gemini_key = getpass.getpass("Enter GEMINI API Key: ")
groq_key = getpass.getpass("Enter GROQ API Key: ")

os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["GROQ_API_KEY"] = groq_key

genai.configure(api_key=gemini_key)
groq_client = Groq(api_key=groq_key)

print("Keys set successfully")

Keys set successfully


In [28]:
corpus = [
    "Transformers use self-attention mechanisms to encode contextual relationships in sequences.",
    "Self-attention computes weighted importance between words in a sequence.",
    "Multi-head attention allows models to focus on different parts of input simultaneously.",
    
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Adam optimizer combines momentum and adaptive learning rates for faster convergence.",
    "Backpropagation computes gradients using the chain rule for training neural networks.",
    
    "Overfitting occurs when a model memorizes training data instead of generalizing.",
    "Dropout randomly deactivates neurons during training to prevent overfitting.",
    "Regularization techniques like L2 penalty help control model complexity.",
    
    "BM25 is a ranking function used in information retrieval based on term frequency.",
    "BERT is a transformer-based model trained using masked language modeling."
]

print("Corpus loaded with", len(corpus), "documents")

Corpus loaded with 11 documents


In [29]:
print("Loading models...")

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Models loaded successfully")

Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully


In [30]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k
        
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)
        
        self.embeddings = sbert_model.encode(corpus)

    def retrieve(self, query, top_k=5):
        print(f"\nRetrieving for query: {query}")
        
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranks = np.argsort(bm25_scores)[::-1]
        
        query_emb = sbert_model.encode([query])[0]
        sbert_scores = np.dot(self.embeddings, query_emb)
        sbert_ranks = np.argsort(sbert_scores)[::-1]
        
        results = []
        
        for i in range(len(self.corpus)):
            bm25_rank = np.where(bm25_ranks == i)[0][0] + 1
            sbert_rank = np.where(sbert_ranks == i)[0][0] + 1
            
            rrf = (1/(self.k+bm25_rank)) + (1/(self.k+sbert_rank))
            
            results.append({
                "doc_id": i,
                "text": self.corpus[i],
                "rrf_score": rrf,
                "bm25_rank": bm25_rank,
                "sbert_rank": sbert_rank
            })
        
        results = sorted(results, key=lambda x: x["rrf_score"], reverse=True)
        
        print("\nTop Retrieved Docs:")
        for r in results[:top_k]:
            print("-", r["text"])
        
        return results[:top_k]

In [31]:
print("\n TESTING RETRIEVER")

retriever = HybridRetriever(corpus)

test_results = retriever.retrieve("how do transformers work?")

print("\nSample Output:")
for r in test_results:
    print(r)


 TESTING RETRIEVER

Retrieving for query: how do transformers work?

Top Retrieved Docs:
- Transformers use self-attention mechanisms to encode contextual relationships in sequences.
- BERT is a transformer-based model trained using masked language modeling.
- BM25 is a ranking function used in information retrieval based on term frequency.
- Gradient descent is an optimization algorithm used to minimize loss functions.
- Backpropagation computes gradients using the chain rule for training neural networks.

Sample Output:
{'doc_id': 0, 'text': 'Transformers use self-attention mechanisms to encode contextual relationships in sequences.', 'rrf_score': 0.03278688524590164, 'bm25_rank': 1, 'sbert_rank': 1}
{'doc_id': 10, 'text': 'BERT is a transformer-based model trained using masked language modeling.', 'rrf_score': 0.03225806451612903, 'bm25_rank': 2, 'sbert_rank': 2}
{'doc_id': 9, 'text': 'BM25 is a ranking function used in information retrieval based on term frequency.', 'rrf_score': 

In [32]:
def rerank(query, docs, top_k=3):
    print("\nRe-ranking documents...")
    
    pairs = [(query, d["text"]) for d in docs]
    scores = cross_encoder.predict(pairs)
    
    for i in range(len(docs)):
        docs[i]["cross_score"] = float(scores[i])
    
    ranked = sorted(docs, key=lambda x: x["cross_score"], reverse=True)
    
    print("\nTop After Re-ranking:")
    for r in ranked[:top_k]:
        print("-", r["text"])
    
    return ranked[:top_k]

In [33]:
print("\nTESTING RE-RANKER")

reranked = rerank("how do transformers work?", test_results)

print("\nFinal Reranked Docs:")
for r in reranked:
    print(r)


TESTING RE-RANKER

Re-ranking documents...

Top After Re-ranking:
- Transformers use self-attention mechanisms to encode contextual relationships in sequences.
- BERT is a transformer-based model trained using masked language modeling.
- Backpropagation computes gradients using the chain rule for training neural networks.

Final Reranked Docs:
{'doc_id': 0, 'text': 'Transformers use self-attention mechanisms to encode contextual relationships in sequences.', 'rrf_score': 0.03278688524590164, 'bm25_rank': 1, 'sbert_rank': 1, 'cross_score': 6.097702980041504}
{'doc_id': 10, 'text': 'BERT is a transformer-based model trained using masked language modeling.', 'rrf_score': 0.03225806451612903, 'bm25_rank': 2, 'sbert_rank': 2, 'cross_score': -9.192102432250977}
{'doc_id': 5, 'text': 'Backpropagation computes gradients using the chain rule for training neural networks.', 'rrf_score': 0.030309988518943745, 'bm25_rank': 7, 'sbert_rank': 5, 'cross_score': -10.774553298950195}


In [40]:
def _select_groq_model():
    preferred = [
        "llama-3.1-8b-instant",
        "llama-3.3-70b-versatile",
        "gemma2-9b-it",
    ]
    
    available = []
    try:
        models = groq_client.models.list()
        available = [m.id for m in getattr(models, "data", [])]
    except Exception:
        pass
    
    for m in preferred:
        if m in available:
            return m
    if available:
        return available[0]
    return preferred[0]

def hyde(query):
    print("\nGenerating hypothetical answer (HyDE)...")
    
    prompt = f"Write a detailed answer for: {query}"
    
    # 1) Try Gemini using dynamically discovered models that support generateContent.
    try:
        preferred = [
            "gemini-2.5-flash",
            "gemini-2.0-flash-lite",
            "gemini-1.5-flash",
            "gemini-1.5-pro",
        ]
        
        available = []
        for m in genai.list_models():
            methods = getattr(m, "supported_generation_methods", [])
            if "generateContent" in methods:
                available.append(m.name.replace("models/", ""))
        
        for model_name in preferred:
            if model_name in available:
                model = genai.GenerativeModel(model_name)
                response = model.generate_content(prompt)
                hypo = response.text
                print(f"Using Gemini model: {model_name}")
                print("\nHypothetical Doc:\n", hypo[:200], "...")
                return hypo
        
        if available:
            model_name = available[0]
            model = genai.GenerativeModel(model_name)
            response = model.generate_content(prompt)
            hypo = response.text
            print(f"Using Gemini model: {model_name}")
            print("\nHypothetical Doc:\n", hypo[:200], "...")
            return hypo
        
        print("No Gemini models with generateContent were available for this API key.")
    except Exception as e:
        print(f"Gemini path failed: {e}")
    
    # 2) Fallback to Groq so the pipeline remains runnable.
    try:
        groq_model = _select_groq_model()
        response = groq_client.chat.completions.create(
            model=groq_model,
            messages=[{"role": "user", "content": prompt}]
        )
        hypo = response.choices[0].message.content
        print(f"Using Groq fallback model: {groq_model}")
        print("\nHypothetical Doc:\n", hypo[:200], "...")
        return hypo
    except Exception as e:
        raise RuntimeError(
            "HyDE generation failed with both Gemini and Groq. "
            "Check API keys, quota, and model access."
        ) from e

In [41]:
print("\n🧪 TESTING HYDE")

hyde("how do transformers encode meaning?")


🧪 TESTING HYDE

Generating hypothetical answer (HyDE)...
Using Gemini model: gemini-2.5-flash

Hypothetical Doc:
 Transformers encode meaning by transforming discrete textual inputs into dense, continuous numerical representations (vectors) that capture semantic, syntactic, and contextual relationships. This proc ...


'Transformers encode meaning by transforming discrete textual inputs into dense, continuous numerical representations (vectors) that capture semantic, syntactic, and contextual relationships. This process is iterative, hierarchical, and highly parallelized, primarily driven by their unique **self-attention mechanism**.\n\nHere\'s a detailed breakdown of how they achieve this:\n\n### 1. Tokenization and Initial Embeddings\n\n*   **Tokenization:** The first step is to break down raw text into smaller units called "tokens." These can be words, subword units (e.g., "un-", "happy", "-ness"), or even individual characters. Subword tokenization (like BPE or WordPiece) is common because it handles rare words, morphology, and reduces vocabulary size while preserving meaning.\n*   **Input Embeddings:** Each token is then mapped to a dense vector in a high-dimensional space. These are the "input embeddings." Initially, these vectors are generic, learned representations where tokens with similar m

In [42]:
def advanced_rag(query):
    print("\n" + "="*60)
    print("ADVANCED RAG PIPELINE")
    print("="*60)
    
    expanded = hyde(query)
    docs = retriever.retrieve(expanded)
    top_docs = rerank(query, docs)
    
    context = "\n".join([d["text"] for d in top_docs])
    
    prompt = f"""
    Answer based on context:
    
    {context}
    
    Question: {query}
    """
    
    last_error = None
    for model_name in ["llama-3.1-8b-instant", "llama-3.3-70b-versatile", _select_groq_model()]:
        try:
            response = groq_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}]
            )
            answer = response.choices[0].message.content
            print(f"Using Groq model: {model_name}")
            print("\nFINAL ANSWER:\n", answer)
            return answer
        except Exception as e:
            last_error = e
            print(f"Model {model_name} failed: {e}")
    
    raise RuntimeError("Advanced RAG answer generation failed on all candidate Groq models.") from last_error

In [43]:
def naive_rag(query):
    print("\nNAIVE RAG")
    
    emb = sbert_model.encode([query])[0]
    scores = np.dot(retriever.embeddings, emb)
    
    idx = np.argmax(scores)
    doc = corpus[idx]
    
    print("Top Doc:", doc)
    
    return doc

In [44]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how to prevent overfitting"
]

results = []

for q in queries:
    naive = naive_rag(q)
    adv = advanced_rag(q)
    results.append((q, naive, adv))

print("\n" + "="*70)
print("COMPARISON TABLE")
print("="*70)

print("\n| Query | Naïve RAG | Advanced RAG | Different? |")
print("|------|-----------|--------------|------------|")

for q, n, a in results:
    diff = "Yes" if n not in a else "No"
    print(f"| {q} | {n[:40]}... | {a[:40]}... | {diff} |")


NAIVE RAG
Top Doc: Transformers use self-attention mechanisms to encode contextual relationships in sequences.

ADVANCED RAG PIPELINE

Generating hypothetical answer (HyDE)...
Using Gemini model: gemini-2.5-flash

Hypothetical Doc:
 Transformers encode meaning by transforming raw text into highly contextualized numerical representations (vectors) through a multi-layered process of attention and transformation. This numerical repr ...

Retrieving for query: Transformers encode meaning by transforming raw text into highly contextualized numerical representations (vectors) through a multi-layered process of attention and transformation. This numerical representation captures the semantic and syntactic relationships between words, phrases, and even entire documents, allowing the model to "understand" and generate human-like text.

Here's a detailed breakdown of how they achieve this:

## 1. The Foundation: Input Representation

Before any "meaning" can be encoded, the raw text needs to be